# Conditional Probability

This notebook introduces **conditional probability** — the probability of an event given that another event has already happened — and generalizes the product rule so it works for *dependent* events too. It is based on the eighth video of Week 1 (Introduction to Probability).


## 1. Introduction

Say you wonder about the probability that today is humid. That is some number. But if you find out that **yesterday it rained**, the probability changes. That updated probability is a **conditional probability**.

Conditioning captures how new information reshapes uncertainty — the core idea behind Bayes' theorem and much of machine learning.

After completing this notebook you will be able to:

- Compute conditional probabilities by shrinking the sample space.
- Read and use the notation $P(B \mid A)$ ("probability of $B$ given $A$").
- Apply the **general product rule** $P(A \cap B) = P(A) \cdot P(B \mid A)$, and see the independent case as a special case.


## 2. Theory

### Conditioning shrinks the sample space

When you learn that event $A$ happened, outcomes incompatible with $A$ are discarded. The remaining outcomes form a **new, smaller sample space**, and probabilities are recomputed inside it:

$$
P(B \mid A) = \frac{\text{outcomes in } B \text{ and } A}{\text{outcomes in } A}
$$

The vertical bar reads "given": $P(B \mid A)$ is the probability of $B$ given that $A$ occurred.

### The general product rule

For any two events — independent or not —

$$
P(A \cap B) = P(A) \cdot P(B \mid A)
$$

### Independence is the special case

If $A$ and $B$ are independent, knowing $A$ tells you nothing about $B$, so $P(B \mid A) = P(B)$ and the formula reduces to the product rule from the previous notebook:

$$
P(A \cap B) = P(A) \cdot P(B) \qquad \text{(independent events only)}
$$

The general rule works **every time**; the simple product is only valid under independence.


## 3. Worked Examples

### Example 1: Two heads, given the first is heads

Throw two coins. The sample space is $\{HH, HT, TH, TT\}$ and $P(HH) = \frac{1}{4}$.

Now suppose you learn that **the first coin landed heads**. The outcomes starting with tails are gone; the new sample space is $\{HH, HT\}$ with 2 outcomes, and $HH$ is still favorable:

$$
P(HH \mid \text{1st is } H) = \frac{1}{2}
$$

### Example 2: Two heads, given the first is tails

If the first coin landed **tails**, the new sample space is $\{TH, TT\}$ — and it contains no $HH$ at all:

$$
P(HH \mid \text{1st is } T) = \frac{0}{2} = 0
$$

Of course: if one coin is tails, both cannot be heads. The condition moved the probability from $\frac{1}{4}$ all the way to $0$.

### Example 3: The general product rule with dice

Roll two dice. What is $P(\text{1st is } 6 \;\cap\; \text{sum is } 10)$?

**Direct count:** of the 36 pairs, only $(6, 4)$ works: $\frac{1}{36}$.

**Product rule:** $P(\text{1st is } 6) = \frac{6}{36}$. Given the first die is 6, the sum is 10 only if the second is 4 — one of the 6 possible outcomes in the shrunken sample space, so $P(\text{sum } 10 \mid \text{1st is } 6) = \frac{1}{6}$. Multiply:

$$
P(\text{1st is } 6 \cap \text{sum } 10) = \frac{6}{36} \cdot \frac{1}{6} = \frac{1}{36} \checkmark
$$

### Example 4: How a condition changes P(sum = 10)

Unconditionally, three pairs sum to 10 — $(4,6), (5,5), (6,4)$ — so $P(\text{sum } 10) = \frac{3}{36} = \frac{1}{12}$.

- **Given the first die is 6**: we live in the bottom row of the table — 6 outcomes, one favorable $(6,4)$: probability **$\frac{1}{6}$** (went up).
- **Given the first die is 1**: we live in the top row — the largest possible sum is $1 + 6 = 7$, so the probability is **$0$** (went down).

The same event can become more likely or impossible depending on the information received.


## 4. Python Examples


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import product

rng = np.random.default_rng(seed=42)


### A helper for conditional probability by counting

The definition translates directly into code: count outcomes satisfying both events, divide by outcomes satisfying the condition.


In [ ]:
def conditional_p(sample_space, event, condition):
    given = [o for o in sample_space if condition(o)]
    favorable = [o for o in given if event(o)]
    return len(favorable), len(given)

two_coins = list(product("HT", repeat=2))

num, den = conditional_p(two_coins, lambda o: o == ("H", "H"),
                         lambda o: o[0] == "H")
print(f"P(HH | 1st is H) = {num}/{den} = {num / den}")

num, den = conditional_p(two_coins, lambda o: o == ("H", "H"),
                         lambda o: o[0] == "T")
print(f"P(HH | 1st is T) = {num}/{den} = {num / den}")


### Examples 3 and 4 in code: dice


In [ ]:
two_dice = list(product(range(1, 7), repeat=2))

# General product rule: P(1st=6 ∩ sum=10) = P(1st=6) * P(sum=10 | 1st=6)
p_first6 = sum(1 for o in two_dice if o[0] == 6) / 36
num, den = conditional_p(two_dice, lambda o: sum(o) == 10, lambda o: o[0] == 6)
p_joint = p_first6 * (num / den)
direct = sum(1 for o in two_dice if o[0] == 6 and sum(o) == 10) / 36
print(f"P(1st=6) * P(sum=10 | 1st=6) = {p_first6:.4f} * {num}/{den} = {p_joint:.4f}")
print(f"Direct count of (6,4):        1/36 = {direct:.4f}\n")

# How conditioning changes P(sum = 10)
print(f"P(sum=10)            = 3/36 = {3/36:.4f}")
for first in [6, 1]:
    num, den = conditional_p(two_dice, lambda o: sum(o) == 10,
                             lambda o, f=first: o[0] == f)
    print(f"P(sum=10 | 1st={first})   = {num}/{den} = {num / den:.4f}")


### Simulating conditional probability

Conditioning in a simulation means **filtering**: keep only the trials where the condition happened, and measure the event's frequency among those.


In [ ]:
n_trials = 500_000
dice = rng.integers(1, 7, size=(n_trials, 2))
sums = dice.sum(axis=1)

first_is_6 = dice[:, 0] == 6
p_conditional = (sums[first_is_6] == 10).mean()

print(f"Trials where 1st die = 6: {first_is_6.sum():,}")
print(f"Simulated P(sum=10 | 1st=6): {p_conditional:.4f}   (theory: {1/6:.4f})")
print(f"Simulated P(sum=10):         {(sums == 10).mean():.4f}   (theory: {1/12:.4f})")


## 5. Visualizations

### Conditioning shrinks the sample space

Left: all four outcomes of two coins, with $HH$ as the event. Right: after learning the first coin is heads, only the top row survives — the event is now 1 of 2 outcomes instead of 1 of 4.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

outcomes = [("H", "H"), ("H", "T"), ("T", "H"), ("T", "T")]
positions = {("H", "H"): (0, 1), ("H", "T"): (1, 1),
             ("T", "H"): (0, 0), ("T", "T"): (1, 0)}

for ax, keep_first_h, title in [
    (axes[0], False, "Before: P(HH) = 1/4"),
    (axes[1], True, "Given 1st is H: P(HH | 1st H) = 1/2"),
]:
    for o in outcomes:
        x, y = positions[o]
        discarded = keep_first_h and o[0] == "T"
        if o == ("H", "H"):
            color = "#e15759"
        elif discarded:
            color = "#eeeeee"
        else:
            color = "#d8e6f2"
        rect = plt.Rectangle((x, y), 1, 1, facecolor=color,
                             edgecolor="white", linewidth=3)
        ax.add_patch(rect)
        ax.text(x + 0.5, y + 0.5, "".join(o), ha="center", va="center",
                fontsize=16, alpha=0.25 if discarded else 1.0)
    ax.set_xlim(0, 2)
    ax.set_ylim(0, 2)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_aspect("equal")
    ax.set_title(title)

plt.suptitle("Learning the first coin's result discards half the sample space")
plt.tight_layout()
plt.show()


### The dice table: conditioning on the first die

Rows are the first die, columns the second. Sum-of-10 cells are red. Conditioning on "first die is 6" restricts us to the bottom row (highlighted): there, 1 of 6 cells is red. Conditioning on "first die is 1" restricts to the top row: no red cells at all.


In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 6.5))

for first in range(1, 7):
    for second in range(1, 7):
        in_row6 = first == 6
        is_sum10 = first + second == 10
        color = "#e15759" if is_sum10 else "#d8e6f2"
        rect = plt.Rectangle((second - 1, 6 - first), 1, 1, facecolor=color,
                             edgecolor="white", linewidth=2,
                             alpha=1.0 if in_row6 else 0.45)
        ax.add_patch(rect)
        ax.text(second - 0.5, 6 - first + 0.5, f"{first},{second}",
                ha="center", va="center", fontsize=9,
                alpha=1.0 if in_row6 else 0.5)

ax.add_patch(plt.Rectangle((0, 0), 6, 1, fill=False,
                           edgecolor="black", linewidth=2.5))
ax.set_xlim(-0.1, 6.1)
ax.set_ylim(-0.1, 6.1)
ax.set_xticks([])
ax.set_yticks([])
ax.set_aspect("equal")
ax.set_title("Given 1st die = 6 (bottom row): P(sum=10) = 1/6, not 3/36")
plt.show()


### The whole story on one page

Everything above in a single sheet: the coin and dice sample spaces shrinking under a condition, how P(sum 10) moves with the first die, the filtering simulation, and the general product rule.

<img src="img/08_conditional_probability.png" alt="Concept sheet for this notebook" width="900">

## 6. Exercises

### Basic

1. You flip two coins. What is $P(\text{2nd is } H \mid \text{1st is } H)$? What does the answer tell you about the two flips?
2. You roll one die. What is $P(6 \mid \text{the roll is even})$?

### Intermediate

3. You roll two dice. What is $P(\text{sum} = 7 \mid \text{1st die is } 3)$?
4. Using the general product rule, compute the probability that the first die is even **and** the sum is 7.

### Advanced

5. You draw **two cards without replacement** from a 52-card deck. Use the general product rule to compute the probability that both are aces, and verify by simulation. (The draws are *not* independent — that's the point.)


## 7. Solutions


In [ ]:
# Exercise 1: the flips are independent, so the condition changes nothing
print("1.  P(2nd H | 1st H) = 1/2 — same as P(2nd H): the flips are independent.")

# Exercise 2: even outcomes {2,4,6}; among them one is a six
print("2.  P(6 | even) = 1/3 =", round(1 / 3, 4))

# Exercise 3: given 1st = 3, sum is 7 only if 2nd = 4
print("3.  P(sum=7 | 1st=3) = 1/6 =", round(1 / 6, 4))

# Exercise 4: P(1st even) = 1/2; given any first value, sum=7 needs one
# specific second value, so P(sum=7 | 1st even) = 1/6
print("4.  P(1st even ∩ sum=7) = 1/2 * 1/6 =", round(1 / 2 * 1 / 6, 4))

# Exercise 5: P(ace, ace) = 4/52 * 3/51
p_two_aces = (4 / 52) * (3 / 51)

deck = np.arange(52)          # 0-3 are the aces
draws = np.array([rng.choice(deck, size=2, replace=False) for _ in range(20_000)])
simulated = np.all(draws < 4, axis=1).mean()
print(f"5.  P(both aces) = 4/52 * 3/51 = {p_two_aces:.5f}, "
      f"simulated = {simulated:.5f}")


## 8. Conclusion

- $P(B \mid A)$ is the probability of $B$ **given** that $A$ happened — computed by shrinking the sample space to the outcomes where $A$ holds.
- Conditions can raise a probability ($\frac{1}{12} \to \frac{1}{6}$), lower it, or destroy it entirely ($\to 0$).
- The **general product rule** $P(A \cap B) = P(A) \cdot P(B \mid A)$ works for *all* events; independence is the special case where $P(B \mid A) = P(B)$.
- In simulations, conditioning means filtering trials — a pattern worth remembering, since it is exactly how Bayes' theorem will be visualized next.
